# **Learn OpenUSD: Learning About Stages, Prims and Attributes**

Welcome to the Jupyter notebook for *Learn OpenUSD: Learning About Stages, Prims and Attributes*. This is where we will find all the Python activities related to this course. Before starting **Activity 1**, make sure to run the cell below.

>**NOTE**: Before starting make sure to run the cell below. This will install the relevant OpenUSD libraries that will be used through this notebook.

In [ ]:
from utils.visualization import DisplayUSD
from utils.helperfunctions import create_new_stage

---

## **Activity 1**: Create a USD File

At its core, an OpenUSD [stage](https://openusd.org/release/glossary.html#usdglossary-stage) refers to a top-level USD file that serves as a container for organizing a hierarchy of elements called prims. Stages aren't files, but a unified scenegraph populated from multiple data sources called [layers](https://openusd.org/release/glossary.html#usdglossary-layer).

Some of the functions we will use to access the stage will be the following:
- [`Usd.Stage.CreateNew()`](https://openusd.org/release/api/class_usd_stage.html#a50c3f0a412aee9decb010787e5ca2e3e): Creates a new empty USD Stage where 3D scenes are assembled.
- [`Usd.Stage.Open()`](https://openusd.org/release/api/class_usd_stage.html#ad3e185c150ee38ae13fb76115863d108): Opens an existing USD file as a stage.
- [`Usd.Stage.Save()`](https://openusd.org/release/api/class_usd_stage.html#adefa2f7ebfc4d8c09f0cd54419aa36c4): Saves the current stage of a USD stage back to a file. If there are multiple layers in the stage, all edited layers that contribute to the stage are being saved. In our case, all edits are being done in a single layer.

**Add the following code to the cell below, then run the cell:**
   
```python
# Import the `Usd` module from the `pxr` package:
from pxr import Usd

# Define a file path name:
file_path = "assets/first_stage.usda"
# Create a stage at the given `file_path`:
stage: Usd.Stage = Usd.Stage.CreateNew(file_path)
print(stage.ExportToString())
```

In [ ]:
# ADD CODE BELOW HERE
# vvvvvvvvvvvvvvvvvvv

# [...]

# ^^^^^^^^^^^^^^^^^^^^
# ADD CODE ABOVE HERE


Here we created a `usda` file using Python. `.usda` are human-readable UTF-8 text. The [Crate file](https://openusd.org/release/glossary.html#crate-file-format) format is USD's own binary file format whose file extension is `.usdc` and is bi-directionally convertible to the `.usda` text format. `.usd` can refer to either Crate or text files. 

Since nothing is in our stage we do not get much from the output.

---

## **Activity 2**: Defining a Cube in the Stage

With the stage created, we can start to add [prims](https://openusd.org/release/glossary.html#usdglossary-prim) and create our scene.

**Add the following code to the cell below, then run the cell:**
   
```python
# Define a prim of type `Cube` at path `/hello`:
UsdGeom.Cube.Define(stage, "/hello")
# Save the stage:
stage.Save()
```

In [ ]:
from pxr import Usd, UsdGeom

file_path = "assets/first_stage.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)

# ADD CODE BELOW HERE
# vvvvvvvvvvvvvvvvvvv

# [...]

# ^^^^^^^^^^^^^^^^^^^^
# ADD CODE ABOVE HERE

DisplayUSD(file_path, show_usd_code=True)

Notice that we're using [`Usd.Stage.Open()`](https://openusd.org/release/api/class_usd_stage.html#ad3e185c150ee38ae13fb76115863d108) rather than [`Usd.Stage.CreateNew()`](https://openusd.org/release/api/class_usd_stage.html#a50c3f0a412aee9decb010787e5ca2e3e). This is because we already created the `.usda` file. 

Here we have defined our first prim. Prims are the primary container object in USD. 

---

## **Activity 3**: Creating a Hierarchy

Prims can contain other prims to create a [namespace hierarchy](https://openusd.org/release/glossary.html#usdglossary-namespace).

**Add the following code to the cell below, then run the cell:**
   
```python
# Define a `Scope` Prim in stage at `/Geometry`:
geom_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, "/Geometry")
# Define an `Xform` Prim in stage as a child of `geom_scope` called `GroupTransform`:
xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, geom_scope.GetPath().AppendPath("GroupTransform"))
# Define a `Cube` in the stage as a child of `xform`, called `Box`:
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, xform.GetPath().AppendPath("Box"))
```

In [ ]:
from pxr import Usd, UsdGeom

file_path = "assets/second_stage.usda"
stage: Usd.Stage = create_new_stage(file_path)

# ADD CODE BELOW HERE
# vvvvvvvvvvvvvvvvvvv

# [...]

# ^^^^^^^^^^^^^^^^^^^^
# ADD CODE ABOVE HERE

stage.Save()
DisplayUSD(file_path, show_usd_code=True)

The hierarchy that we have created is:

- Geometry
    - GroupTransform
        - Cube
     
In this example, `Geometry` is the root prim. We are also defining three types of prims here: [`scope`](https://openusd.org/release/api/class_usd_geom_scope.html), [`xform`](https://openusd.org/release/api/class_usd_geom_xform.html), and [`cube`](https://openusd.org/release/api/class_usd_geom_cube.html).

- `Xform` defines a transform (translate, rotation, scale).
- `Scope` is a simple container that does not hold transform data.
- `Cube` defines a primitive rectilinear cube.

---

## **Activity 4**: Lighting a Stage

So far we have created prims using [`UsdGeom`](https://openusd.org/release/api/usd_geom_page_front.html). This is a [`schema`](https://openusd.org/release/glossary.html#usdglossary-schema) that defines 3D graphics-related prim and property schemas. USD also comes with other schemas, like [`UsdLux`](https://openusd.org/release/api/usd_lux_page_front.html) which provides a representation for lights and related components.

We're going to define two new prims: [`SphereLight`](https://openusd.org/dev/api/class_usd_lux_sphere_light.html) and [`DistantLight`](https://openusd.org/release/api/class_usd_lux_distant_light.html)

**Add the following code to the cell below, then run the cell:**
   
```python
# Define a `Scope` Prim in stage at `/Lights`:
lights_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, "/Lights")
# Define a `Sun` prim in stage as a child of `lights_scope`, called `Sun`:
distant_light = UsdLux.DistantLight.Define(stage, lights_scope.GetPath().AppendPath("Sun"))
# Define a `SphereLight` prim in stage as a child of lights_scope called `SphereLight`:
sphere_light = UsdLux.SphereLight.Define(stage, lights_scope.GetPath().AppendPath("SphereLight"))

# Configure the distant light's emissive attributes:
distant_light.GetColorAttr().Set(Gf.Vec3f(1.0, 0.0, 0.0)) # Light color (red)
distant_light.GetIntensityAttr().Set(120.0) # Light intensity
# Position the distant light in the 3D scene:
distant_light_transform = distant_light.GetTransformOp()
if not distant_light_transform:
    distant_light_transform = distant_light.AddTransformOp()
distant_light_transform.Set(Gf.Matrix4d((pi/4, 0, -pi/4, 0), (0, 1, 0, 0), (pi/4, 0, pi/4, 0), (10, 0, 10, 1)))
distant_light.GetXformOpOrderAttr().Set([distant_light_transform.GetName()])

# Configure the sphere light's emissive attributes:
sphere_light.GetColorAttr().Set(Gf.Vec3f(0.0, 0.0, 1.0)) # Light color (blue)
sphere_light.GetIntensityAttr().Set(50000.0) # Light intensity
# Position the sphere light in the 3D scene:
sphere_light_transform = sphere_light.GetTransformOp()
if not sphere_light_transform:
    sphere_light_transform = sphere_light.AddTransformOp()
sphere_light_transform.Set(Gf.Matrix4d((pi/4, 0, pi/4, 0), (0, 1, 0, 0), (-pi/4, 0, pi/4, 0), (-10, 0, 10, 1)))
sphere_light.GetXformOpOrderAttr().Set([sphere_light_transform.GetName()])
```

In [ ]:
from math import pi
from pxr import Gf, Usd, UsdGeom, UsdLux

file_path = "assets/second_stage.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)
geom_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, "/Geometry")
xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, geom_scope.GetPath().AppendPath("GroupTransform"))
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, xform.GetPath().AppendPath("Box"))

# ADD CODE BELOW HERE
# vvvvvvvvvvvvvvvvvvv

# [...]

# ^^^^^^^^^^^^^^^^^^^^
# ADD CODE ABOVE HERE

stage.Save()
DisplayUSD(file_path, show_usd_code=True, show_usd_lights=True)

Now our hierarchy looks like the following:

- Geometry
    - GroupTransform
        - Box
- Lights
    - Sun
    - SphereLight

We have introduced two new prims: [`UsdLux.SphereLight`](https://openusd.org/dev/api/class_usd_lux_sphere_light.html) and [`UsdLux.DistantLight`](https://openusd.org/release/api/class_usd_lux_distant_light.html).

---

## **Activity 5**: Adding Attributes to a Prim

[`Attributes`](https://openusd.org/release/glossary.html#attribute) are the most common type of property authored in most USD scenes. An attribute consist of a type and default value. It can also include time sampled values.

Here is an example of a `Cube` prim with an attribute called `weight`:

```python
def Cube "Box" 
{
    double weight = 50
}
```

**Add the following code to the cell below, then run the cell:**
   
```python
# Get the Cube prim at path: `/Geometry/GroupTransform/Box`:
cube: UsdGeom.Cube = UsdGeom.Cube(stage.GetPrimAtPath("/Geometry/GroupTransform/Box"))
# Get all attribute names associated with Cube:
cube_attrs = cube.GetSchemaAttributeNames()
# Print out all the attribute names for Cube:
for attr in cube_attrs:
    print(attr)
# Get the Attribute for "Display Color":
cube_color_attr: Usd.Attribute = cube.GetDisplayColorAttr()
# Set the "Display Color" attribute to red:
cube_color_attr.Set([(1.0, 0.0, 0.0)])
```

In [ ]:
from pxr import Usd, UsdGeom

file_path = "assets/second_stage.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)

# ADD CODE BELOW HERE
# vvvvvvvvvvvvvvvvvvv

# [...]

# ^^^^^^^^^^^^^^^^^^^^
# ADD CODE ABOVE HERE

stage.Save()
DisplayUSD(file_path, show_usd_code=True)

To see all attributes defined by a schema we used [`GetSchemaAttributeNames()`](https://openusd.org/release/api/class_usd_geom_cube.html#a7c62fe4bca24edb5beae756618bf602f). Each schema will contain their own predetermined attributes.

After retrieving the attribute we can set the value using [`Set()`](https://openusd.org/release/api/class_usd_attribute.html#a7fd0957eecddb7cfcd222cccd51e23e6). Getting the attribute does not mean we are retrieving the value of the attribute.

---

## **Activity 6**: Getting the Value of a Current Attribute

Each schema has their own set of attributes and corresponding functions to retrieve them. To view more, look at [`UsdGeomCube`'s](https://openusd.org/release/api/class_usd_geom_cube.html#pub-methods) public member functions. The function names typically end with *Attr*. 

Let's look into setting another attribute and how we can get the value from an attribute.


**Add the following code to the cell below, then run the cell:**
   
```python
# Get the Cube's `size` attribute:
cube_size_attr: Usd.Attribute = cube.GetSizeAttr()
# Get the value of the Cube's `Size` attribute, double it and set it as the new value for the `Size` attribute:
cube_size_attr.Set(cube_size_attr.Get() * 2)
# Get the prim at the path: `/Geometry/GroupTransform`:
geom_scope: UsdGeom.Scope = stage.GetPrimAtPath("/Geometry/GroupTransform")
# Define a Cube Prim in stage as a child of geom_scope called `Small_Box`:
small_box: UsdGeom.Cube = UsdGeom.Cube.Define(stage, geom_scope.GetPath().AppendPath("Small_Box"))
# Set the position of the `small_box` to x: 4, y: 5, z: 4
UsdGeom.XformCommonAPI(small_box).SetTranslate((4, 5, 4))
```

In [ ]:
from pxr import Usd, UsdGeom

file_path = "assets/second_stage.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)

cube: UsdGeom.Cube = UsdGeom.Cube(stage.GetPrimAtPath("/Geometry/GroupTransform/Box"))
cube_color_attr: Usd.Attribute = cube.GetDisplayColorAttr()
cube_color_attr.Set([(1.0, 0.0, 0.0)])

# ADD CODE BELOW HERE
# vvvvvvvvvvvvvvvvvvv

# [...]

# ^^^^^^^^^^^^^^^^^^^^
# ADD CODE ABOVE HERE

stage.Save()
DisplayUSD(file_path, show_usd_code=True)

Here we used `GetSizeAttr()` to retrieve the attribute that defines the size of the cube. This is also a public member function defined in `UsdGeomCube`. To get the value of an attribute we use [`Get()`](https://openusd.org/release/api/class_usd_attribute.html#a9d41bc223be86408ba7d7f74df7c35a9). Each [`UsdAttribute`](https://openusd.org/release/api/class_usd_attribute.html) contains a `Set()` and `Get()` function for its values. 

[`XformCommonAPI`](https://openusd.org/release/api/class_usd_geom_xform_common_a_p_i.html) is used to set and get transform components such as scale, rotation, scale-rotate pivot and translation. Even though these are considered attributes, it is best to go through `XformCommonAPI` when editting transformation values. `XformCommonAPI` is a great way to bootstrap setting up new transformations. Future courses will dive into advanced usage of xformOps. Below is an example to check if `XformCommonAPI` is compatible with the prim.

``` python 
from pxr import Usd, UsdGeom

# Create a stage and define a prim path
stage = Usd.Stage.CreateNew('example.usda')
prim = UsdGeom.Xform.Define(stage, '/ExamplePrim')

# Check if the XformCommonAPI is compatible with the prim using the bool operator 
if not (xform_api := UsdGeom.XformCommonAPI(prim)):
    raise Exception("Prim not compatible with XformCommonAPI")

# Set transformations
xform_api.SetTranslate((10.0, 20.0, 30.0))
xform_api.SetRotate((45.0, 0.0, 90.0), UsdGeom.XformCommonAPI.RotationOrderXYZ)
xform_api.SetScale((2.0, 2.0, 2.0))

```

---

## **Activity 7**: Traversing a Stage

We can traverse a USD stage using [`Usd.Stage.Traverse()`](https://openusd.org/release/api/class_usd_stage.html#adba675b55f41cc1b305bed414fc4f178). This is fundamental when working with USD files, especially in complex 3D scenes and pipelines.

**Add the following code to the cell below, then run the cell:**
   
```python
for prim in stage.Traverse():
    print(prim.GetPath())
```

In [ ]:
from pxr import Usd

file_path = "assets/second_stage.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)

# ADD CODE BELOW HERE
# vvvvvvvvvvvvvvvvvvv

# [...]

# ^^^^^^^^^^^^^^^^^^^^
# ADD CODE ABOVE HERE

Traversing is done depth-first search and can be used to not just go throught the whole stage but a branch of the whole tree.

---

## **Activity 8**: Does the Prim Exist?

When working with large amounts of data it is key to make sure that a prim exists before trying to author it. We can get the child of a prim using [`GetChild()`](https://openusd.org/release/api/class_usd_prim.html#a8c0974bbd49570564f0096ce982ff64a). If it was unable to find the child, it will return an invalid `UsdPrim`. You can use [`Usd.Object.IsValid()`](https://openusd.org/release/api/class_usd_object.html#ac532c4b500b1a85ea22217f2c65a70ed) to check if the prim is valid or exists. 

**Add the following code to the cell below, then run the cell:**
   
```python
prim: Usd.Prim = stage.GetPrimAtPath("/Geometry")
child_prim: Usd.Prim
if child_prim := prim.GetChild("Box"):
    print("Child prim exists")
else:
    print("Child prim DOES NOT exist")
```

In [ ]:
from pxr import Usd

file_path = "assets/second_stage.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)

# ADD CODE BELOW HERE
# vvvvvvvvvvvvvvvvvvv

# [...]

# ^^^^^^^^^^^^^^^^^^^^
# ADD CODE ABOVE HERE

`Box` is not a child of `Geometry`. If we change `Box` to `GroupTransform` then it will print out "Child prim exists". That is because `GetChild()` retrieves the direct child of the prim, not nested children.